# Notebook 04 — Computer Vision: Genre Classification with CLIP

**Project:** Concert Ticket Price Predictor  
**Block:** Computer Vision  

This notebook evaluates the CV block:
- **Model:** `openai/clip-vit-base-patch32` (zero-shot image classification via HF API)
- **Data:** Wikipedia artist thumbnail images fetched via REST API
- **Task:** Classify artist/concert photos into music genres
- **Integration:** Predicted genre → pre-fills the genre dropdown in the ML Price Predictor

The same Wikipedia REST API used in Notebook 02 (for bio text) also provides thumbnail images,
demonstrating that Source 2 (Wikipedia) contributes to both the NLP and CV blocks.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..') / 'src'))

import io
import time
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
from PIL import Image

from cv_classifier import (
    GENRE_PROMPTS, GENRE_LABELS,
    classify_genre, fetch_artist_thumbnail, preprocess_image
)

## 1. Image Preprocessing Pipeline

In [ ]:
print('CLIP zero-shot labels and prompts:')
print()
for genre, prompt in GENRE_PROMPTS.items():
    print(f'  {genre:20s} → "{prompt}"')

print()
print('Preprocessing pipeline:')
print('  1. Load image (upload / URL / bytes)')
print('  2. Convert to RGB')
print('  3. Resize to 224x224 (LANCZOS resampling)')
print('  4. Encode to JPEG bytes for API call')
print('  5. Send to HF Inference API (openai/clip-vit-base-patch32)')
print('  6. CLIP computes cosine similarity between image embedding')
print('     and each genre prompt text embedding')
print('  7. Softmax over similarities → genre probabilities')

## 2. Fetch Artist Thumbnails (Source 2: Wikipedia API)

In [ ]:
# Evaluation set: 12 artists with known genres
EVAL_SET = [
    ('Taylor Swift',     'Rock'),
    ('Metallica',        'Metal'),
    ('Eminem',           'Hip-Hop/Rap'),
    ('Imagine Dragons',  'Rock'),
    ('Adele',            'R&B'),
    ('The Chainsmokers', 'Dance/Electronic'),
    ('Luke Bryan',       'Country'),
    ('Jason Aldean',     'Country'),
    ('Post Malone',      'Hip-Hop/Rap'),
    ('Ed Sheeran',       'Pop'),
    ('Shawn Mendes',     'Pop'),
    ('Godsmack',         'Rock'),
]

thumbnails = {}
for artist, genre in EVAL_SET:
    raw = fetch_artist_thumbnail(artist)
    thumbnails[artist] = raw
    status = 'OK' if raw else 'MISS'
    print(f'[{status}] {artist} ({genre})')
    time.sleep(0.3)

found = sum(1 for v in thumbnails.values() if v)
print(f'\nFetched: {found}/{len(EVAL_SET)} thumbnails')

## 3. Visual Inspection of Thumbnails

In [ ]:
artists_with_img = [(a, g) for a, g in EVAL_SET if thumbnails.get(a)]
n = len(artists_with_img)
cols = 4
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
axes = axes.flatten()

for i, (artist, true_genre) in enumerate(artists_with_img):
    img = Image.open(io.BytesIO(thumbnails[artist])).convert('RGB')
    axes[i].imshow(img)
    axes[i].set_title(f'{artist}\n(genre: {true_genre})', fontsize=9)
    axes[i].axis('off')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Artist Thumbnails from Wikipedia (Source 2)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../data/plot_cv_thumbnails.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. CLIP Genre Classification

In [ ]:
print('Running CLIP zero-shot classification...')
print('(Requires HF_TOKEN — skipped if unavailable)\n')

results = []
for artist, true_genre in EVAL_SET:
    raw = thumbnails.get(artist)
    if not raw:
        results.append({'artist': artist, 'true_genre': true_genre,
                        'pred_genre': 'N/A', 'confidence': 0.0, 'correct': False})
        continue

    pred_genre, conf = classify_genre(raw)
    if pred_genre is None:
        pred_genre, conf = 'API unavailable', 0.0

    correct = pred_genre == true_genre
    symbol = '✓' if correct else '✗'
    print(f'[{symbol}] {artist:22s} true={true_genre:18s} pred={pred_genre:18s} conf={conf:.2%}')
    results.append({'artist': artist, 'true_genre': true_genre,
                    'pred_genre': pred_genre, 'confidence': conf, 'correct': correct})
    time.sleep(1.0)  # respect rate limits

df_results = pd.DataFrame(results)
valid = df_results[df_results['pred_genre'] != 'API unavailable']
if len(valid) > 0:
    acc = valid['correct'].mean()
    print(f'\nAccuracy: {acc:.0%} ({valid["correct"].sum()}/{len(valid)} correct)')

In [ ]:
print('Results summary:')
print(df_results[['artist', 'true_genre', 'pred_genre', 'confidence']].to_string(index=False))

## 5. Evaluation & Limitations

**Evaluation metrics:** Top-1 accuracy on 12 artists with known genres.

**Expected behavior:**
- CLIP works well for visually distinct genres (Metal bands look different from Country singers)
- CLIP may confuse Pop and R&B (both have similar visual aesthetics)
- CLIP struggles when the Wikipedia thumbnail is a headshot (no visual genre cues)
- Zero-shot classification without fine-tuning gives reasonable but not perfect results

**Known limitations:**
1. Wikipedia thumbnails vary in quality — some are headshots with no genre-specific visual cues
2. Genre boundaries are subjective (e.g. Taylor Swift is officially Pop/Country but listed as Rock in training data)
3. CLIP was trained on web-crawled images; artist-specific visual style is not always consistent
4. Zero-shot accuracy (~50–70% expected) is lower than a fine-tuned model on dedicated artist images

**Integration with ML block:**
The genre prediction from CLIP pre-fills the genre dropdown in the Price Predictor app.
The user can correct it before running the ML model, making CV a helpful starting point.
This is documented in `src/cv_classifier.py` and `app.py`.